# Exercise for Unit 4.1 — Naïve Bayes

---

## Dataset

In [ ]:
# ── Dataset ──────────────────────────────────────────────────────────────────
documents = [
    ("Free money now!!!",                    "SPAM"),
    ("Hi mom, how are you?",                 "HAM"),
    ("Lowest price for your meds",           "SPAM"),
    ("Are we still on for dinner?",          "HAM"),
    ("Win a free iPhone today",              "SPAM"),
    ("Let's catch up tomorrow at the office","HAM"),
    ("Meeting at 3 PM tomorrow",             "HAM"),
    ("Get 50% off, limited time!",           "SPAM"),
    ("Team meeting in the office",           "HAM"),
    ("Click here for prizes!",               "SPAM"),
    ("Can you send the report?",             "HAM"),
]

test_sentences = [
    "Limited offer, click here!",
    "Meeting at 2 PM with the manager.",
]

---
## Part 1 — Manual Naïve Bayes Implementation

### 1-a. Bag of Words (word frequency per class)

In [ ]:
import re
from collections import defaultdict

def tokenize(text):
    """Lowercase and extract alphabetic tokens."""
    return re.findall(r'[a-z]+', text.lower())

def build_bag_of_words(documents):
    """
    Returns:
        bow      : dict[class] -> dict[word] -> count
        vocab    : set of all unique words
        class_doc_counts : dict[class] -> number of documents
    """
    bow = defaultdict(lambda: defaultdict(int))
    class_doc_counts = defaultdict(int)

    for text, label in documents:
        class_doc_counts[label] += 1
        for word in tokenize(text):
            bow[label][word] += 1

    vocab = set()
    for label_counts in bow.values():
        vocab.update(label_counts.keys())

    return dict(bow), vocab, dict(class_doc_counts)

bow, vocab, class_doc_counts = build_bag_of_words(documents)

print("=== Bag of Words ===")
for label in sorted(bow):
    print(f"\n[{label}]")
    for word, cnt in sorted(bow[label].items()):
        print(f"  {word:15s}: {cnt}")

print(f"\nVocabulary size : {len(vocab)}")
print(f"Vocabulary      : {sorted(vocab)}")

### 1-b. Prior Probabilities P(HAM) and P(SPAM)

In [ ]:
def calculate_priors(class_doc_counts):
    """P(class) = (# docs in class) / (total docs)"""
    total = sum(class_doc_counts.values())
    return {label: count / total for label, count in class_doc_counts.items()}

priors = calculate_priors(class_doc_counts)

print("=== Prior Probabilities ===")
total_docs = sum(class_doc_counts.values())
for label, prior in sorted(priors.items()):
    print(f"  P({label}) = {class_doc_counts[label]}/{total_docs} = {prior:.4f}")

### 1-c. Likelihood P(word | class) with Laplace Smoothing

In [ ]:
def calculate_likelihoods(bow, vocab, alpha=1):
    """
    Laplace-smoothed likelihood:
        P(word | class) = (count(word, class) + alpha)
                          / (total_words_in_class + alpha * |vocab|)
    """
    likelihoods = {}
    V = len(vocab)

    for label, word_counts in bow.items():
        total_words = sum(word_counts.values())
        likelihoods[label] = {}
        for word in vocab:
            count = word_counts.get(word, 0)
            likelihoods[label][word] = (count + alpha) / (total_words + alpha * V)

    return likelihoods

likelihoods = calculate_likelihoods(bow, vocab)

print("=== Likelihoods P(word | class) — Laplace smoothed (alpha=1) ===")
for label in sorted(likelihoods):
    print(f"\n[{label}]")
    for word, prob in sorted(likelihoods[label].items()):
        print(f"  P({word:15s} | {label}) = {prob:.6f}")

### 1-d. Classify Test Sentences

In [ ]:
import math

def classify(text, priors, likelihoods, vocab, alpha=1):
    """
    Naïve Bayes classifier using log-probabilities to avoid underflow.
    Unknown words receive the smoothed probability for an unseen token.
    """
    tokens = tokenize(text)
    scores = {}

    for label in priors:
        # Start with log prior
        log_prob = math.log(priors[label])
        total_words_in_class = sum(bow[label].values())
        V = len(vocab)

        for token in tokens:
            if token in likelihoods[label]:
                log_prob += math.log(likelihoods[label][token])
            else:
                # Unseen word — use Laplace smoothed probability
                log_prob += math.log(alpha / (total_words_in_class + alpha * V))

        scores[label] = log_prob

    predicted = max(scores, key=scores.get)
    return predicted, scores

print("=== Classification Results (Manual Naïve Bayes) ===")
for sentence in test_sentences:
    predicted, scores = classify(sentence, priors, likelihoods, vocab)
    print(f"\nSentence : \"{sentence}\"")
    for label in sorted(scores):
        print(f"  log P({label}) = {scores[label]:.6f}")
    print(f"  → Predicted class : {predicted}")

---
## Part 2 — Multinomial Naïve Bayes using Scikit-Learn

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

# Separate texts and labels
texts  = [doc for doc, _ in documents]
labels = [lbl for _, lbl in documents]

# Build pipeline: CountVectorizer → MultinomialNB
pipeline = Pipeline([
    ('vectorizer', CountVectorizer()),
    ('classifier', MultinomialNB()),
])

# Train on the full dataset
pipeline.fit(texts, labels)

print("=== Classification Results (Scikit-Learn MultinomialNB) ===")
for sentence in test_sentences:
    prediction = pipeline.predict([sentence])[0]
    proba      = pipeline.predict_proba([sentence])[0]
    classes    = pipeline.classes_
    print(f"\nSentence : \"{sentence}\"")
    for cls, p in zip(classes, proba):
        print(f"  P({cls}) = {p:.6f}")
    print(f"  → Predicted class : {prediction}")